## Press Play Button to Run the program

In [ ]:
# === Interactive Image Upload, Prediction & Grad-CAM with PDF Report ===
import os
import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms as T
import torchvision.models as models
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display, FileLink
from ipywidgets import (FileUpload, Output, VBox, HBox, Label, 
                        Text, Dropdown, Button, Layout, HTML)
import io
from report import generate_report

# ----------------------------
# CONFIGURATION & MODEL SETUP
# ----------------------------
MODEL_PATH = "assets/best_model_224_resnet50.pt"
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE = (224, 224)

# Build and load model
print("Loading model...")
model = models.resnet50(pretrained=False)
in_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Linear(in_features, 128),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(128, 1)
)
model = model.to(DEVICE)

checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)
if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
    state = checkpoint['model_state_dict']
else:
    state = checkpoint
from collections import OrderedDict
clean_state = OrderedDict()
for k, v in state.items():
    new_k = k[len('module.'):] if k.startswith('module.') else k
    clean_state[new_k] = v
model.load_state_dict(clean_state)
model.eval()
print(f"Model loaded on {DEVICE}")

# ----------------------------
# GRAD-CAM IMPLEMENTATION
# ----------------------------
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        def forward_hook(module, input, output):
            self.activations = output.detach()
        def backward_hook(module, grad_in, grad_out):
            self.gradients = grad_out[0].detach()
        target_layer.register_forward_hook(forward_hook)
        target_layer.register_backward_hook(backward_hook)

    def __call__(self, input_tensor):
        output = self.model(input_tensor)
        loss = output[0, 0]
        self.model.zero_grad()
        loss.backward(retain_graph=True)
        grads = self.gradients[0]
        acts = self.activations[0]
        weights = grads.mean(dim=(1, 2))
        cam = (weights[:, None, None] * acts).sum(dim=0)
        cam = F.relu(cam)
        cam = cam - cam.min()
        if cam.max() != 0:
            cam = cam / cam.max()
        return cam.cpu().numpy()

# Initialize Grad-CAM
target_layer = model.layer4[-1].conv3
gradcam = GradCAM(model, target_layer)

# Preprocessing pipeline
preprocess = T.Compose([
    T.Resize(IMG_SIZE),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ----------------------------
# UNIFIED PREDICTION & GRADCAM FUNCTION
# ----------------------------
def process_image_with_gradcam(image_bytes):
    """
    Load image from bytes, run prediction and Grad-CAM visualization
    Returns results dict and gradcam overlay image array
    """
    try:
        # Load image from bytes
        img_pil = Image.open(io.BytesIO(image_bytes)).convert('RGB')
        print(f"Image loaded: {img_pil.size}")
        
        # Preprocess for model
        img_tensor = preprocess(img_pil).unsqueeze(0).to(DEVICE)
        
        # Prediction
        with torch.no_grad():
            output = model(img_tensor)
            raw_logit = output.item()
            prob_non_cancer = torch.sigmoid(torch.tensor(raw_logit)).item()
            prob_cancer = 1 - prob_non_cancer
            pred_label = "NON-CANCER" if prob_non_cancer >= 0.5 else "CANCER"
        
        print(f"Prediction: {pred_label}")
        print(f"  Prob CANCER:     {prob_cancer:.4f}")
        print(f"  Prob NON-CANCER: {prob_non_cancer:.4f}")
        
        # Grad-CAM heatmap
        cam = gradcam(img_tensor)
        heatmap = cv2.resize(cam, IMG_SIZE)
        heatmap = np.uint8(255 * heatmap)
        heatmap_colored = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
        
        # Original image to numpy
        img_np = cv2.cvtColor(np.array(img_pil.resize(IMG_SIZE)), cv2.COLOR_RGB2BGR)
        
        # Overlay
        superimposed = cv2.addWeighted(img_np, 0.6, heatmap_colored, 0.4, 0)
        
        # Convert BGR to RGB for display
        superimposed_rgb = cv2.cvtColor(superimposed, cv2.COLOR_BGR2RGB)
        
        # Visualization
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        
        # Original
        axes[0].imshow(cv2.cvtColor(img_np, cv2.COLOR_BGR2RGB))
        axes[0].set_title('Original Image')
        axes[0].axis('off')
        
        # Heatmap
        axes[1].imshow(heatmap, cmap='hot')
        axes[1].set_title('Grad-CAM Heatmap')
        axes[1].axis('off')
        
        # Overlay with prediction
        axes[2].imshow(superimposed_rgb)
        axes[2].set_title(f'Grad-CAM Overlay\nPrediction: {pred_label}\nConfidence: {max(prob_cancer, prob_non_cancer):.4f}')
        axes[2].axis('off')
        
        plt.tight_layout()
        plt.show()
        
        return {
            'label': pred_label,
            'prob_cancer': prob_cancer,
            'prob_non_cancer': prob_non_cancer,
            'image': img_pil,
            'gradcam_overlay': superimposed_rgb
        }
        
    except Exception as e:
        print(f"Error processing image: {str(e)}")
        return None

# ----------------------------
# DEMOGRAPHICS COLLECTION WIDGETS
# ----------------------------
print("=" * 60)
print("ORAL CANCER SCREENING SYSTEM")
print("=" * 60)

# Create demographics widgets
demographics_output = Output()

age_input = Text(
    value='',
    placeholder='e.g., 45',
    description='Age:',
    style={'description_width': '180px'},
    layout=Layout(width='300px')
)

gender_input = Dropdown(
    options=['Male', 'Female', 'Other', 'Prefer not to say'],
    value='Male',
    description='Gender:',
    style={'description_width': '180px'},
    layout=Layout(width='300px')
)

smoking_input = Dropdown(
    options=['Never', 'Former smoker', 'Current smoker (< 10/day)', 
             'Current smoker (10-20/day)', 'Current smoker (> 20/day)'],
    value='Never',
    description='Smoking Habit:',
    style={'description_width': '180px'},
    layout=Layout(width='400px')
)

alcohol_input = Dropdown(
    options=['Never', 'Occasional (< 1 drink/week)', 'Moderate (1-7 drinks/week)', 
             'Heavy (> 7 drinks/week)', 'Daily'],
    value='Never',
    description='Alcohol Consumption:',
    style={'description_width': '180px'},
    layout=Layout(width='400px')
)

tobacco_input = Dropdown(
    options=['Never', 'Former user', 'Current user (occasional)', 
             'Current user (daily)', 'Heavy user'],
    value='Never',
    description='Tobacco/Betel Chewing:',
    style={'description_width': '180px'},
    layout=Layout(width='400px')
)

submit_demographics_btn = Button(
    description='Submit Demographics',
    button_style='success',
    layout=Layout(width='200px', height='40px')
)

# Storage for demographics
user_demographics = {}

# ----------------------------
# IMAGE UPLOAD WIDGET
# ----------------------------
uploader = FileUpload(
    accept='image/*',
    multiple=False,
    description='Upload Image',
    button_style='info',
    disabled=True  # Initially disabled
)

output_area = Output()
pdf_download_area = Output()

def on_submit_demographics(btn):
    """Handle demographics submission"""
    demographics_output.clear_output(wait=True)
    with demographics_output:
        if not age_input.value or not age_input.value.isdigit():
            print("⚠️ Please enter a valid age (numeric value)")
            return
        
        user_demographics['age'] = int(age_input.value)
        user_demographics['gender'] = gender_input.value
        user_demographics['smoking'] = smoking_input.value
        user_demographics['alcohol'] = alcohol_input.value
        user_demographics['tobacco_chewing'] = tobacco_input.value
        
        print("✓ Demographics saved successfully!")
        print("\nYou can now upload an image for analysis.")
        
        # Enable image upload
        uploader.disabled = False
        
        # Disable demographics editing
        age_input.disabled = True
        gender_input.disabled = True
        smoking_input.disabled = True
        alcohol_input.disabled = True
        tobacco_input.disabled = True
        submit_demographics_btn.disabled = True

submit_demographics_btn.on_click(on_submit_demographics)

def on_upload_change(change):
    """Handle image upload and processing"""
    output_area.clear_output(wait=True)
    pdf_download_area.clear_output(wait=True)
    
    with output_area:
        if len(uploader.value) > 0:
            if not user_demographics:
                print("⚠️ Please submit demographics first!")
                return
            
            uploaded_file = uploader.value[0]
            file_content = uploaded_file['content']
            print(f"Processing {uploaded_file['name']}...")
            print("-" * 60)
            
            result = process_image_with_gradcam(file_content)
            
            if result:
                print("\n✓ Image processing completed successfully!")
                print("=" * 60)
                print("Generating PDF report...")
                
                try:
                    # Generate PDF report
                    pdf_path = generate_report(
                        model_results={
                            'label': result['label'],
                            'prob_cancer': result['prob_cancer'],
                            'prob_non_cancer': result['prob_non_cancer']
                        },
                        demographics_data=user_demographics,
                        gradcam_image_array=result['gradcam_overlay']
                    )
                    
                    print("=" * 60)
                    print("✓ REPORT GENERATION COMPLETE!")
                    print("=" * 60)
                    
                    # Display download link
                    with pdf_download_area:
                        print("\n📄 Your PDF report is ready!")
                        display(FileLink(pdf_path, result_html_prefix="Click here to download: "))
                        
                except Exception as e:
                    print(f"\n⚠️ Error generating PDF report: {str(e)}")
                    print("Model results are still displayed above.")
        else:
            print("No file uploaded yet.")

uploader.observe(on_upload_change, names='value')

# ----------------------------
# DISPLAY INTERFACE
# ----------------------------
print("\nSTEP 1: Please provide your demographic information")
print("-" * 60)
display(VBox([
    Label(value='Patient Demographics:', 
          layout=Layout(margin='10px 0 10px 0')),
    age_input,
    gender_input,
    smoking_input,
    alcohol_input,
    tobacco_input,
    submit_demographics_btn,
    demographics_output
], layout=Layout(padding='10px')))

print("\n" + "=" * 60)
print("STEP 2: Upload image for analysis (enabled after demographics)")
print("-" * 60)
display(VBox([
    Label(value='Upload an oral cavity image:'),
    uploader,
    output_area,
    pdf_download_area
], layout=Layout(padding='10px')))


Loading model...
Model loaded on cpu
ORAL CANCER SCREENING SYSTEM

STEP 1: Please provide your demographic information
------------------------------------------------------------



STEP 2: Upload image for analysis (enabled after demographics)
------------------------------------------------------------
